# BioServices Overview

**BioServices** is a Python package that provides programmatic access to many bioinformatics web services, including UniProt, Ensembl, KEGG, NCBI EUtils, ChEMBL, and more. Instead of manually navigating web browsers or writing HTTP requests by hand, BioServices provides clean Python classes that wrap each service's API.

This notebook introduces the core concept and demonstrates three commonly used services:
- **UniProt** — the comprehensive protein sequence and functional information database
- **Ensembl** — the genome browser and annotation database
- **KEGG** — the Kyoto Encyclopedia of Genes and Genomes, covering metabolic pathways

Each service is imported individually rather than using a wildcard import, which keeps the namespace clean and makes the code self-documenting.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

---
## 1. UniProt

[UniProt](https://www.uniprot.org/) is the world's leading high-quality, comprehensive and freely accessible resource of protein sequence and functional information. It is split into:
- **Swiss-Prot** (reviewed, manually annotated)
- **TrEMBL** (unreviewed, automatically annotated)

The `UniProt` class in BioServices allows you to search for proteins, retrieve sequences in FASTA format, and download structured data as a pandas DataFrame.

In [ ]:
from bioservices import UniProt

u = UniProt(verbose=False)

### 1.1 Searching for proteins

The `search()` method queries the UniProt REST API. The query syntax is the same as the UniProt website search box. Here we search for the human ZAP-70 kinase, a key signalling molecule in T-cell activation.

In [ ]:
# Search for ZAP70 in Homo sapiens (taxonomy ID 9606), limit to 5 results
results = u.search("ZAP70 AND taxonomy_id:9606", limit=5)
print(results)

### 1.2 Retrieving a protein FASTA sequence

Given a UniProt accession number, `get_fasta()` returns the protein sequence in FASTA format. UniProt accession **P43403** is ZAP-70 (Zeta-chain-associated protein kinase 70) from *Homo sapiens*.

In [ ]:
fasta = u.get_fasta("P43403")
print(fasta)

### 1.3 Retrieving structured data as a DataFrame

`get_df()` fetches a set of UniProt fields for one or more accession IDs and returns a pandas DataFrame, making it easy to inspect annotations, PDB structures, PTMs, and more.

In [ ]:
df = u.get_df("P43403")
# Show a small selection of the columns
cols = ["Entry", "Entry Name", "Organism", "Gene Names (primary)", "Protein names"]
print(df[cols].to_string())

---
## 2. Ensembl

[Ensembl](https://www.ensembl.org/) is a genome browser and annotation database maintained by EMBL-EBI and the Wellcome Sanger Institute. It provides gene, transcript, and variant annotations for a large number of species.

The `Ensembl` class in BioServices wraps the Ensembl REST API. We can look up genes by their stable Ensembl IDs (e.g. `ENSG...` for genes, `ENST...` for transcripts) or by gene symbol.

In [ ]:
from bioservices import Ensembl

e = Ensembl(verbose=False)

### 2.1 Looking up a gene by Ensembl ID

`get_lookup_by_id()` returns a dictionary of annotations for the given stable identifier. Here we look up **ENSG00000157764**, which is the *BRAF* gene (B-Raf proto-oncogene, serine/threonine kinase) in humans. The `expand=True` argument includes transcript information in the response.

In [ ]:
# Look up the BRAF gene
braf = e.get_lookup_by_id('ENSG00000157764', expand=True)
print("Gene:", braf.get('display_name'))
print("Description:", braf.get('description'))
print("Location: chr", braf.get('seq_region_name'),
      braf.get('start'), '-', braf.get('end'),
      '(strand', braf.get('strand'), ')')
print("Number of transcripts:", len(braf.get('Transcript', [])))

### 2.2 Listing available species

`get_info_species()` returns metadata about all species supported by Ensembl. We can filter this list to find, for example, all ovine (sheep) species entries.

In [ ]:
species_data = e.get_info_species()
species_list = species_data.get('species', [])

# Filter for human-readable names that contain 'homo'
human_entries = [s for s in species_list if 'homo' in s.get('name', '').lower()]
for s in human_entries:
    print(s['name'], '-', s.get('display_name', ''), '-', s.get('assembly', ''))

---
## 3. KEGG

[KEGG](https://www.kegg.jp/) (Kyoto Encyclopedia of Genes and Genomes) is a database resource for understanding high-level functions and utilities of the biological system. It is especially well known for its manually drawn **pathway maps** that represent molecular interaction, reaction, and relation networks.

The `KEGG` class in BioServices wraps the KEGG REST API and lets us search for pathways, genes, compounds, and more.

In [ ]:
from bioservices import KEGG

k = KEGG(verbose=False)

### 3.1 Searching for pathways

`find()` searches the KEGG pathway database for entries matching a keyword. Here we look for pathways related to B-cell signalling, which is relevant to immune responses and certain cancers.

In [ ]:
# Search KEGG pathways for 'B cell'
results = k.find('pathway', 'B cell')
print(results)

### 3.2 Fetching a specific pathway entry

`get()` retrieves the full KEGG record for a given entry ID. Here we fetch the human B-cell receptor signalling pathway (`hsa04662`). The result is a flat text file with structured fields.

In [ ]:
# Fetch the pathway record (first 600 characters)
entry = k.get('hsa04662')
print(entry[:600])

---
## Summary

In this notebook we covered three core BioServices services:

| Service   | What it does | Key methods used |
|-----------|-------------|------------------|
| UniProt   | Protein sequences & functional annotation | `search()`, `get_fasta()`, `get_df()` |
| Ensembl   | Genome annotation & comparative genomics | `get_lookup_by_id()`, `get_info_species()` |
| KEGG      | Metabolic/signalling pathways & gene functions | `find()`, `get()` |

BioServices supports many more services — see the [full documentation](https://bioservices.readthedocs.io/) for a complete list.